In [34]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# prep
iris = load_iris()
X = iris.data
y = iris.target
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test = jnp.array(X_train), jnp.array(X_test)
y_train, y_test = jnp.array(y_train), jnp.array(y_test)

In [36]:
# model
class SimpleNN(nn.Module):
	def setup(self):
		self.dense1 = nn.Dense(16)
		self.dense2 = nn.Dense(3)

	@jax.jit
	def __call__(self, x):
		x = self.dense1(x)
		x = nn.relu(x)
		x = self.dense2(x)
		return x

# loss function (cross entropy)
@jax.jit
def compute_loss(params, X, y):
	logits = model.apply({'params': params}, X)
	loss = optax.softmax_cross_entropy(logits, jax.nn.one_hot(y, num_classes=3)).mean()
	return loss

# update params
@jax.jit
def update(params, opt_state, X, y):
	grads = jax.grad(compute_loss)(params, X, y)
	updates, new_opt_state = optimizer.update(grads, opt_state)
	new_params = optax.apply_updates(params, updates)
	return new_params, new_opt_state

In [37]:
# define
model = SimpleNN()
params = model.init(jax.random.PRNGKey(42), X_train[:1])['params']
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(params)

# train
epochs = 100
for epoch in range(epochs):
	params, opt_state = update(params, opt_state, X_train, y_train)

	if (epoch + 1) % 10 == 0 or epoch == 0:
		loss = compute_loss(params, X_train, y_train)
		print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss:.4f}")

Epoch [1/100], Loss: 0.8882
Epoch [10/100], Loss: 0.4954
Epoch [20/100], Loss: 0.3613
Epoch [30/100], Loss: 0.2925
Epoch [40/100], Loss: 0.2448
Epoch [50/100], Loss: 0.2045
Epoch [60/100], Loss: 0.1668
Epoch [70/100], Loss: 0.1343
Epoch [80/100], Loss: 0.1089
Epoch [90/100], Loss: 0.0919
Epoch [100/100], Loss: 0.0804


In [38]:
# evaluate
logits = model.apply({'params': params}, X_test)
y_pred = jax.nn.softmax(logits, axis=1)
accuracy = jnp.mean(jnp.argmax(y_pred, axis=1) == y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 100.00%
